# Week 18, Day 4: Insertion sort and bubble sort

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w018d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
hand = [3, 7, 9]
card = 5
hand.append(card)
i = len(hand) - 1
while i > 0 and hand[i - 1] > hand[i]:
    hand[i - 1], hand[i] = hand[i], hand[i - 1]
    i -= 1
print(hand)

In [ ]:
def insertion_sort(items):
    for i in range(1, len(items)):
        current = items[i]                 # lift it out
        j = i - 1
        while j >= 0 and items[j] > current:
            items[j + 1] = items[j]        # shift the bigger item right
            j -= 1
        items[j + 1] = current             # drop it into the gap

data = [5, 2, 9, 1, 7]
insertion_sort(data)
print(data)

*Interactive figure on the website: Insertion sort: compare the counts on sorted, nearly sorted and reversed data. Then try bubble sort on the same data..*

In [ ]:
def insertion_counting(items):
    items = items[:]
    comparisons = moves = 0
    for i in range(1, len(items)):
        current = items[i]
        j = i - 1
        while j >= 0:
            comparisons += 1
            if items[j] <= current:
                break
            items[j + 1] = items[j]
            moves += 1
            j -= 1
        items[j + 1] = current
    return comparisons, moves

import random
n = 200
print("sorted  ", insertion_counting(list(range(n))))
print("reversed", insertion_counting(list(range(n, 0, -1))))
print("random  ", insertion_counting(random.Random(1).sample(range(n), n)))
nearly = list(range(n))
nearly[50], nearly[51] = nearly[51], nearly[50]
print("nearly  ", insertion_counting(nearly))

In [ ]:
def bubble_sort(items):
    n = len(items)
    for end in range(n - 1, 0, -1):         # the last 'end' places are being fixed
        for j in range(end):
            if items[j] > items[j + 1]:
                items[j], items[j + 1] = items[j + 1], items[j]

data = [5, 2, 9, 1, 7]
bubble_sort(data)
print(data)

In [ ]:
def bubble_sort(items):
    passes = 0
    for end in range(len(items) - 1, 0, -1):
        passes += 1
        swapped = False
        for j in range(end):
            if items[j] > items[j + 1]:
                items[j], items[j + 1] = items[j + 1], items[j]
                swapped = True
        if not swapped:
            break                            # nothing moved: it's sorted
    return passes

print(bubble_sort(list(range(10))), "pass on sorted data")
print(bubble_sort([1, 2, 3, 5, 4, 6, 7, 8, 9, 10]), "passes on nearly sorted data")
print(bubble_sort(list(range(10, 0, -1))), "passes on reversed data")

In [ ]:
def inversions(items):
    return sum(1 for i in range(len(items)) for j in range(i + 1, len(items)) if items[i] > items[j])

def insertion_moves(items):
    items = items[:]
    moves = 0
    for i in range(1, len(items)):
        current = items[i]
        j = i - 1
        while j >= 0 and items[j] > current:
            items[j + 1] = items[j]
            moves += 1
            j -= 1
        items[j + 1] = current
    return moves

for data in ([3, 1, 2], [5, 2, 9, 1, 7], list(range(10, 0, -1))):
    print(data, inversions(data), insertion_moves(data))

> **Key idea.** Insertion sort takes each item in turn and slides it left into place: O(n) on sorted data, O(n²) worst case, stable, and the best simple sort in practice. Bubble sort swaps neighbours in passes; stop when a pass makes no swaps. Both do one move per inversion.

### Exercise: Insertion sort

Write `insertion_sort(items)` that sorts a list in place using the lift-shift-drop method and returns nothing. Don't use `sort` or `sorted`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    for data in ([5, 2, 9, 1, 7], [], [1], [2, 1], [3, 3, 1, 3], list(range(10, 0, -1)), [\"kiwi\", \"fig\", \"apple\"]):\n        copy = list(data)\n        assert insertion_sort(copy) is None and copy == sorted(data), data\n    rng = random.Random(2)\n    data = [rng.randint(-50, 50) for _ in range(400)]\n    copy = list(data)\n    insertion_sort(copy)\n    assert copy == sorted(data)\n    assert \"sorted\" not in source and \".sort(\" not in source")

### Exercise: Insert into a sorted list

Write `insert_sorted(items, x)` that adds `x` to an already sorted list, in place, in the right position, by appending it and then shifting it left (one step of insertion sort). Don't use `sort`, `sorted`, `insert` or `bisect`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    data = [1, 4, 6, 9]\n    insert_sorted(data, 5)\n    assert data == [1, 4, 5, 6, 9]\n    insert_sorted(data, 0)\n    insert_sorted(data, 10)\n    assert data == [0, 1, 4, 5, 6, 9, 10]\n    empty = []\n    insert_sorted(empty, 3)\n    assert empty == [3]\n    assert \"sort\" not in source.replace(\"insert_sorted\", \"\") and \"insert(\" not in source.replace(\"insert_sorted(\", \"\") and \"bisect\" not in source")

### Exercise: Bubble sort with an early stop

Write `bubble_sort(items)` that sorts a list in place and **returns the number of passes** it made, stopping as soon as a pass makes no swaps. Each pass should stop before the items already bubbled into place at the end.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    data = [5, 2, 9, 1, 7]\n    assert bubble_sort(data) == 4 and data == [1, 2, 5, 7, 9]\n    data = list(range(10))\n    assert bubble_sort(data) == 1 and data == list(range(10))\n    data = [2, 1, 3, 4, 5]\n    assert bubble_sort(data) == 2 and data == [1, 2, 3, 4, 5]\n    assert bubble_sort([]) == 0 and bubble_sort([1]) == 0\n    data = list(range(20, 0, -1))\n    assert bubble_sort(data) == 19 and data == list(range(1, 21))")

### Exercise: Moves equal inversions

Write `insertion_moves(items)` that returns how many times insertion sort shifts an item right when sorting a copy of `items`, and `inversions(items)` that counts pairs `i < j` with `items[i] > items[j]` directly with two loops. Then set `reversed_20` to the number of moves insertion sort makes on `list(range(20, 0, -1))`, worked out from a formula.

In [ ]:
reversed_20 = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    assert insertion_moves([3, 1, 2]) == 2 and inversions([3, 1, 2]) == 2\n    assert insertion_moves([]) == 0 and inversions([1, 2, 3]) == 0\n    rng = random.Random(4)\n    for _ in range(30):\n        data = [rng.randint(0, 9) for _ in range(rng.randint(0, 25))]\n        assert insertion_moves(data) == inversions(data)\n    assert reversed_20 == 190")

### Exercise: Stable by a key

Write `insertion_sort_by(items, key)` that sorts in place by `key(item)`, keeping items with equal keys in their original order. Then sort `STUDENTS` (name, year) by year using it, so students in the same year stay in alphabetical order (they already are).

In [ ]:
STUDENTS = [("Ada", 2), ("Ben", 1), ("Cy", 2), ("Dee", 1), ("Eve", 3), ("Fin", 1)]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"stable\"\n    assert STUDENTS == [(\"Ben\", 1), (\"Dee\", 1), (\"Fin\", 1), (\"Ada\", 2), (\"Cy\", 2), (\"Eve\", 3)]\n    words = [\"bb\", \"a\", \"cc\", \"d\", \"ee\"]\n    insertion_sort_by(words, len)\n    assert words == [\"a\", \"d\", \"bb\", \"cc\", \"ee\"]\n    assert \"sorted\" not in source and \".sort(\" not in source")

### Bug hunt: Off the front

This insertion sort crashes with `IndexError` on most lists: when an item belongs at the very front, `j` keeps going below 0. Fix the loop condition.

In [ ]:
def insertion_sort(items):
    for i in range(1, len(items)):
        current = items[i]
        j = i - 1
        while items[j] > current:
            items[j + 1] = items[j]
            j -= 1
        items[j + 1] = current

data = [3, 1, 2, 4, 0]
insertion_sort(data)
print(data)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"sorted\"\n    expect_output(\"[0, 1, 2, 3, 4]\")")

### Bug hunt: Off the end

This bubble sort crashes with `IndexError`. Fix the inner loop's range.

In [ ]:
def bubble_sort(items):
    for end in range(len(items) - 1, 0, -1):
        for j in range(end + 1):
            if items[j] > items[j + 1]:
                items[j], items[j + 1] = items[j + 1], items[j]

data = [4, 3, 1, 2]
bubble_sort(data)
print(data)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"sorted\"\n    expect_output(\"[1, 2, 3, 4]\")")

### Puzzle: Insertion sort

Arrange the lines into insertion sort. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # the list after the first PASS of bubble sort on [4, 1, 3, 2]
b = ___   # the list after insertion sort has inserted the first 3 items of [4, 1, 3, 2] (i up to 2)
c = ___   # comparisons insertion sort makes on an already sorted list of 50 items
d = ___   # shifts insertion sort makes on [2, 1, 4, 3]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == [1, 3, 2, 4], \"a\"\n    assert b == [1, 3, 4, 2], \"b\"\n    assert c == 49 and d == 2")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: keeping a leaderboard sorted

A game keeps a leaderboard of the top scores, highest first, and new scores arrive one at a time. Re-sorting the whole board after every score is wasteful: the board is already sorted, so one step of insertion sort is enough. Write two functions:

- `add_score(board, name, score, size=10)`: insert `(name, score)` into `board` (a list of pairs sorted by score from highest to lowest) in place, after any existing equal scores (earlier scores keep their rank), then cut the board to `size` entries. Return the new entry's position (0 is top) or `None` if it didn't make the board. Shift items like insertion sort; don't use `sort`, `sorted`, `insert` or `bisect`.
- `ranks(board)`: a list of strings like `"1. Ada 980"`, numbered from 1.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_add():\n    \"add_score\"\n    board = []\n    assert add_score(board, \"Ada\", 50, size=3) == 0\n    assert add_score(board, \"Ben\", 70, size=3) == 0\n    assert add_score(board, \"Cy\", 50, size=3) == 2\n    assert board == [(\"Ben\", 70), (\"Ada\", 50), (\"Cy\", 50)]\n    assert add_score(board, \"Dee\", 40, size=3) is None and len(board) == 3\n    assert add_score(board, \"Eve\", 60, size=3) == 1\n    assert board == [(\"Ben\", 70), (\"Eve\", 60), (\"Ada\", 50)]\ndef test_ranks():\n    \"ranks\"\n    assert ranks([(\"Ben\", 70), (\"Eve\", 60)]) == [\"1. Ben 70\", \"2. Eve 60\"] and ranks([]) == []\ndef test_many():\n    \"many\"\n    rng = random.Random(5)\n    board = []\n    entries = [(f\"p{i}\", rng.randint(0, 1000)) for i in range(2000)]\n    for name, score in entries:\n        add_score(board, name, score, size=10)\n    order = sorted(range(len(entries)), key=lambda i: (-entries[i][1], i))[:10]\n    assert board == [entries[i] for i in order]\n    assert \"sort\" not in source and \"bisect\" not in source and \".insert(\" not in source")

## Warm-up

A new card joins a sorted hand at the right, then slides left past every bigger card until it meets a smaller one. Do that for each card in turn, starting with a hand of one, and you've sorted the whole lot: **insertion sort**.

## Insertion sort

Like selection sort, insertion sort keeps a sorted part on the left. But instead of finding the right **item** for the next place, it takes the next item and finds the right **place** for it in the sorted part:

```
[5 | 2, 9, 1, 7]      take 2: it's smaller than 5, so 5 moves right
[2, 5 | 9, 1, 7]      take 9: it's bigger than 5, stays
[2, 5, 9 | 1, 7]      take 1: 9, 5 and 2 each move right; 1 goes at the front
[1, 2, 5, 9 | 7]      take 7: 9 moves right; 7 goes after 5
[1, 2, 5, 7, 9]
```

Rather than swapping the item left one step at a time, it's neater (and does fewer writes) to **lift it out**, shift each bigger item one place right, and drop it into the gap:

Notice the `while` condition: `j >= 0` stops it at the front of the list, and `items[j] > current` stops it at the first item that isn't bigger. Put the bounds check first: `and` stops at the first false part, so `items[j]` is never looked at once `j` is -1. Leave the bounds check out and `j` keeps going below 0, reading `items[-1]`, `items[-2]` and so on from the far end, until it crashes with `IndexError`.

Using `>` rather than `>=` means an item never moves past an equal one, which makes insertion sort **stable**: equal items keep their original order.

(The figure shows insertion sort as swaps, so you can see each item move; the lifted version does the same comparisons with fewer writes.)

## Analysing insertion sort

How far does each item travel? It depends on the data.

- **Best case, already sorted:** each item is compared once with its left neighbour, isn't smaller, and stays. n − 1 comparisons, no moves: **O(n)**.
- **Worst case, reversed:** each item travels all the way to the front: 1 + 2 + ... + (n − 1) = n(n − 1)/2 comparisons and moves: **O(n²)**.
- **Average, random order:** each item travels about halfway: about n²/4, still **O(n²)**.

That's insertion sort's great strength: on **nearly sorted** data it's close to O(n). Data is often nearly sorted in practice: a sorted list with a few new items added at the end, readings that drift slowly. It's also very fast on **short** lists, because each step is so simple. For both reasons, Python's own sort (Timsort, Week 19) uses insertion sort on short runs of up to about 64 items.

## Bubble sort

Bubble sort only ever compares **neighbours**. It walks along the list comparing each pair, swapping any that are the wrong way round. After one pass, the biggest item has been carried ("bubbled") all the way to the end. After the second pass, the second biggest is in place, and so on.

As written, bubble sort always does n(n − 1)/2 comparisons, like selection sort, but makes far more swaps: every swap moves an item just one place. That makes it slower than both of the others in practice.

The one trick that helps: if a whole pass makes **no swaps**, every neighbouring pair is in order, so the list is sorted and you can stop.

With the early stop, bubble sort's best case is one pass, O(n), like insertion sort. Its worst and average are O(n²). Bubble sort is mainly taught because it's easy to describe; Donald Knuth wrote that it seems to have nothing to recommend it except a catchy name. Insertion sort does everything it does, better.

## Inversions: what these sorts really fix

Remember inversions from last week's Forge: pairs of items in the wrong order. A sorted list has 0 inversions; a reversed list has the most, n(n − 1)/2.

Every swap of two **neighbours** that are in the wrong order fixes **exactly one** inversion. Both bubble sort and insertion sort only ever move items past their neighbours, so:

**The number of swaps (or shifts) that insertion sort and bubble sort make equals the number of inversions in the list.**

That's why they're fast on nearly sorted data (few inversions) and slow on reversed data (the most possible). And it shows a limit: any sort that only swaps neighbours must make one swap per inversion, about n²/4 on random data, so it can never beat O(n²) on average. To do better, a sort has to move items long distances in one go, as merge sort and quicksort do.

## Three sorts side by side

| | Selection | Insertion | Bubble (with early stop) |
|---|---|---|---|
| Idea | pick the smallest left | insert the next into place | swap neighbours, biggest bubbles up |
| Best case | O(n²) | O(n) | O(n) |
| Worst case | O(n²) | O(n²) | O(n²) |
| Swaps or moves | at most n − 1 | one per inversion | one per inversion |
| Stable? | no | yes | yes |
| Extra space | O(1) | O(1) | O(1) |
| Good for | few writes | small or nearly sorted lists | teaching |

## Practice

## Quiz

## Daily challenge

## Before you go

Sort a shuffled hand of cards by insertion sort, then shuffle again and sort by bubble sort, counting the swaps each time. Count the inversions in the starting hand too: the numbers should match. Tomorrow: stability, why it matters, and how to compare sorts properly.